In [ ]:
# | default_exp content_mapper

# Content Mapper
> Map website URLs to local source files using direct, slug, or limax strategies.

In [ ]:
# | export
from pathlib import Path
from seootter.index_tracking import fetch_sitemap_urls
from seootter.gsc_client import GSCAuth



In [ ]:
# | export
SOURCE_EXTS = (".qmd", ".md", ".ipynb", ".mdx", ".astro")


def url_to_relpath(url: str,  # Full page URL
                   base_path: str,  # Local base directory
                   site_url: str  # Site base URL to strip
                   ) -> Path | None:
    "Convert a page URL to a relative local path, stripping site URL and .html suffix."
    url_cleaned = url.removeprefix(site_url).removesuffix(".html").lstrip("/")
    return Path(base_path) / url_cleaned

In [ ]:
# | export


def find_source_file(rel_path: Path,  # Relative path without extension
                     exts: tuple = SOURCE_EXTS  # Extensions to try in order
                     ) -> str | None:
    "Find first existing source file matching the path with any of the given extensions."
    for ext in exts:
        candidate = rel_path.with_suffix(ext)
        if candidate.exists(): return str(candidate)
    return None



In [ ]:
# |export
def url_to_file_path(url: str,  # Full page URL
                     base_path: str,  # Local base directory
                     site_url: str  # Site base URL
                     ) -> str | None:
    "Map a website URL to its local source file path."
    rel_path = url_to_relpath(url, base_path, site_url)
    return find_source_file(rel_path) if rel_path else None


In [ ]:
# | export
def build_limax_slug_index(base_path: str  # Local base directory
                           ) -> dict[str, str]:
    "Build slug→filepath index using Node/limax transliteration for Arabic Astro sites."
    import subprocess, json
    script = """
    const slugify = require('limax');
    const fs = require('fs');
    const path = require('path');
    const dir = process.env.LIMAX_BASE_PATH;
    const exts = ['.md', '.qmd', '.ipynb', '.mdx', '.astro'];
    const index = {};
    function walk(d) {
        for (const f of fs.readdirSync(d)) {
            const full = path.join(d, f);
            if (fs.statSync(full).isDirectory()) { walk(full); continue; }
            if (exts.some(e => f.endsWith(e))) {
                const name = path.basename(f, path.extname(f));
                index[slugify(name)] = full;
            }
        }
    }
    walk(dir);
    console.log(JSON.stringify(index));
    """
    result = subprocess.run(['node', '-e', script], capture_output=True, text=True,
                            cwd=base_path, env={**__import__('os').environ, 'LIMAX_BASE_PATH': base_path})
    if result.returncode != 0: raise RuntimeError(f"limax slug index failed: {result.stderr}")
    return json.loads(result.stdout)


In [ ]:
# | export
def build_slug_index(base_path: str  # Local base directory
                     ) -> dict[str, str]:
    "Build slug→filepath index by reading `slug` field from frontmatter of all source files."
    import yaml
    index = {}
    for ext in SOURCE_EXTS:
        for path in Path(base_path).rglob(f"*{ext}"):
            try:
                content = path.read_text(encoding="utf-8")
                if content.startswith("---"):
                    meta = yaml.safe_load(content.split("---")[1])
                    if slug := (meta or {}).get("slug"):
                        index[slug] = str(path)
            except Exception:
                continue
    return index



In [ ]:
# | export
from seootter.gsc.filters import normalize_url


def map_all_urls_to_files(base_path: str,  # Local base directory
                          site_url: str,  # Site base URL
                          urls: list[str],  # List of page URLs to map
                          mode: str = "direct",  # One of: 'direct', 'slug', 'limax'
                          normalize: bool = False  # Normalize URLs before mapping
                          ) -> dict[str, str]:
    "Map page URLs to local source file paths using the specified strategy."
    if mode not in ("direct", "slug", "limax"):
        raise ValueError(f"Invalid mode {mode!r}. Must be one of: 'direct', 'slug', 'limax'")
    if mode in ("slug", "limax"):
        slug_index = (build_limax_slug_index if mode == "limax" else build_slug_index)(base_path)
        return {url: slug_index[slug] for url in urls
                if (slug := url.removeprefix(site_url).lstrip("/").removesuffix("/")) in slug_index}
    return {normalize_url(url) if normalize else url: path
            for url in urls if (path := url_to_file_path(url, base_path, site_url))}



In [ ]:
# | hide
#| eval: false
from dotenv import load_dotenv
import os

load_dotenv()
# BASE_PATH = os.environ["SEEOOTTER_BASE_PATH"]
BASE_PATH = os.environ["AWAZLY_PATH"]

urls = fetch_sitemap_urls("https://awazly.com/sitemap.xml")
map_all_urls_to_files(
    base_path=f"{BASE_PATH}",
    site_url="https://awazly.com/",
    urls=urls,
    mode="limax"
)
